In [19]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [20]:
load_dotenv()

llm = ChatOpenAI()

In [21]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation : str

In [22]:
def generate_joke(state : JokeState):
    prompt = f"Generate a joke for this given topic \n {state['topic']}"
    response = llm.invoke(prompt).content

    return{'joke' : response}

In [23]:
def explain_joke(state : JokeState):
    prompt = f"Explain the below given joke \n {state['joke']}"
    response = llm.invoke(prompt).content

    return{'explanation' : response}

In [24]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('explain_joke', explain_joke)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer = checkpointer)


In [29]:
config1 = {'configurable' : {'thread_id' : '1'}}

initial_state = {
    'topic' : 'IPL'
}

workflow.invoke(initial_state, config = config1)

{'topic': 'IPL',
 'joke': 'Why did the cricket player go to the bank during the IPL season? \nTo get some fast cash!',
 'explanation': 'This joke plays on the idea of "fast cash" commonly associated with cricket players due to their high salaries during the Indian Premier League (IPL) season. The joke suggests that the cricket player went to the bank during the IPL season because he wanted to get some money quickly, or "fast cash," reflecting the lucrative earnings that players can make during this popular cricket tournament.'}

In [31]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'IPL', 'joke': 'Why did the cricket player go to the bank during the IPL season? \nTo get some fast cash!', 'explanation': 'This joke plays on the idea of "fast cash" commonly associated with cricket players due to their high salaries during the Indian Premier League (IPL) season. The joke suggests that the cricket player went to the bank during the IPL season because he wanted to get some money quickly, or "fast cash," reflecting the lucrative earnings that players can make during this popular cricket tournament.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1358ec-f062-6907-800a-8a83e164841d'}}, metadata={'source': 'loop', 'step': 10, 'parents': {}}, created_at='2026-04-11T10:11:30.011121+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1358ec-e4c2-6d3c-8009-684dae0b1448'}}, tasks=(), interrupts=())

In [32]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'IPL', 'joke': 'Why did the cricket player go to the bank during the IPL season? \nTo get some fast cash!', 'explanation': 'This joke plays on the idea of "fast cash" commonly associated with cricket players due to their high salaries during the Indian Premier League (IPL) season. The joke suggests that the cricket player went to the bank during the IPL season because he wanted to get some money quickly, or "fast cash," reflecting the lucrative earnings that players can make during this popular cricket tournament.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1358ec-f062-6907-800a-8a83e164841d'}}, metadata={'source': 'loop', 'step': 10, 'parents': {}}, created_at='2026-04-11T10:11:30.011121+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1358ec-e4c2-6d3c-8009-684dae0b1448'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'IPL', 'joke': 'Why di

In [34]:
config2 = {'configurable' : {'thread_id' : '2'}}

initial_state = {
    'topic' : 'Tomato'
}

workflow.invoke(initial_state, config = config2)

{'topic': 'Tomato',
 'joke': 'Why did the tomato turn red? Because it saw the salad dressing!',
 'explanation': 'This joke is a play on words using the fact that tomatoes are red when they are ripe. The punchline suggests that the tomato turned red out of embarrassment or shock upon seeing the salad dressing, implying that the tomato is feeling like it is being dressed up or prepared as part of a salad.'}

In [35]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Tomato', 'joke': 'Why did the tomato turn red? Because it saw the salad dressing!', 'explanation': 'This joke is a play on words using the fact that tomatoes are red when they are ripe. The punchline suggests that the tomato turned red out of embarrassment or shock upon seeing the salad dressing, implying that the tomato is feeling like it is being dressed up or prepared as part of a salad.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1358f1-8899-63fb-8002-8be97746ef63'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-11T10:13:33.346072+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1358f1-7fc5-6e71-8001-df582bf8e16a'}}, tasks=(), interrupts=())

In [36]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tomato', 'joke': 'Why did the tomato turn red? Because it saw the salad dressing!', 'explanation': 'This joke is a play on words using the fact that tomatoes are red when they are ripe. The punchline suggests that the tomato turned red out of embarrassment or shock upon seeing the salad dressing, implying that the tomato is feeling like it is being dressed up or prepared as part of a salad.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1358f1-8899-63fb-8002-8be97746ef63'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-11T10:13:33.346072+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1358f1-7fc5-6e71-8001-df582bf8e16a'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Tomato', 'joke': 'Why did the tomato turn red? Because it saw the salad dressing!'}, next=('explain_joke',), config={'configurable': {'thread_id': 

In [40]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1358f1-7985-6194-8000-03c8576a870b"}})

StateSnapshot(values={'topic': 'Tomato'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1358f1-7985-6194-8000-03c8576a870b'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-04-11T10:13:31.764963+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1358f1-7981-6f9f-bfff-99f3081481a4'}}, tasks=(PregelTask(id='229fb188-2081-7211-bb0d-a0a29d36a106', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the tomato turn red? Because it saw the salad dressing!'}),), interrupts=())

In [41]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f1358f1-7985-6194-8000-03c8576a870b"}})

{'topic': 'Tomato',
 'joke': 'Why did the tomato turn red? Because it saw the salad dressing!',
 'explanation': 'This joke is based on the idea that when a tomato is ripe and ready to eat, it turns red. The joke plays on this idea by suggesting that the tomato turned red not because it was ripe, but because it saw the salad dressing and got embarrassed, as if it were blushing. It is a play on words and a light-hearted way to introduce humor into a common everyday situation.'}

In [42]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tomato', 'joke': 'Why did the tomato turn red? Because it saw the salad dressing!', 'explanation': 'This joke is based on the idea that when a tomato is ripe and ready to eat, it turns red. The joke plays on this idea by suggesting that the tomato turned red not because it was ripe, but because it saw the salad dressing and got embarrassed, as if it were blushing. It is a play on words and a light-hearted way to introduce humor into a common everyday situation.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f13591f-ea2e-6aaa-8002-fe2dacf3c44f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-11T10:34:18.381557+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f13591f-df23-68b2-8001-d185c04ec81a'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Tomato', 'joke': 'Why did the tomato turn red? Because it saw the salad dre

In [43]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1358f1-7985-6194-8000-03c8576a870b", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f135922-b306-6b16-8001-1624775af66a'}}

In [44]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f135922-b306-6b16-8001-1624775af66a'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-04-11T10:35:33.128661+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1358f1-7985-6194-8000-03c8576a870b'}}, tasks=(PregelTask(id='2f203b85-f6af-5af4-39b2-14d5d3c9fa06', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Tomato', 'joke': 'Why did the tomato turn red? Because it saw the salad dressing!', 'explanation': 'This joke is based on the idea that when a tomato is ripe and ready to eat, it turns red. The joke plays on this idea by suggesting that the tomato turned red not because it was ripe, but because it saw the salad dressing and got embarrassed, as i

In [45]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f135922-b306-6b16-8001-1624775af66a"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to school? \nBecause it wanted to get a little more filling!',
 'explanation': 'This joke is a play on words. In this context, "filling" can mean both the ingredients inside the samosa and also the knowledge or education that one gains from going to school. So, the joke suggests that the samosa went to school in order to gain more knowledge or education, just like how a person might go to school to learn and grow.'}

In [46]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to school? \nBecause it wanted to get a little more filling!', 'explanation': 'This joke is a play on words. In this context, "filling" can mean both the ingredients inside the samosa and also the knowledge or education that one gains from going to school. So, the joke suggests that the samosa went to school in order to gain more knowledge or education, just like how a person might go to school to learn and grow.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f135926-342c-65a8-8003-05a6e39d33bb'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-04-11T10:37:07.201346+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f135926-2979-6c0e-8002-46fa7ff2ab3c'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to school? \nBecause it wanted to get a litt